In [7]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


# E-commerce Return Analysis and Prediction

This notebook solves an interview task using an e-commerce orders dataset.

The work is divided into three parts:

1. Data Wrangling
2. Exploratory Data Analysis
3. Simple Machine Learning Modeling

## Part 1: Load and Inspect the Dataset

Before cleaning, analyzing, or modeling the data, I first inspect the dataset structure.

This includes:

- dataset shape
- column names
- data types
- missing values
- sample rows

In [8]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [9]:
# Get project root folder
PROJECT_ROOT = Path.cwd()

# If notebook runs from notebooks folder, move one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ecommerce_orders.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

Project root: d:\11.Running Projects\interview-return-analysis
Dataset path: d:\11.Running Projects\interview-return-analysis\data\raw\ecommerce_orders.csv
Dataset exists: True


In [10]:
df = pd.read_csv(DATA_PATH)

df.head()

,order_id,customer_id,order_date,product_category,amount,discount_pct,is_returned
0,ORD_00180,CUST_012,7/1/2023,Books,15.15,25.0,0
1,ORD_00303,CUST_007,7/2/2023,Clothing,117.31,10.0,1
2,ORD_00421,CUST_054,7/2/2023,Electronics,643.68,25.0,0
3,ORD_00163,CUST_005,7/2/2023,Clothing,134.04,10.0,0
4,ORD_00032,CUST_075,7/4/2023,Food,17.79,50.0,0


In [11]:
print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape:
(500, 7)

Column names:
['order_id', 'customer_id', 'order_date', 'product_category', 'amount', 'discount_pct', 'is_returned']

Data types:
order_id                str
customer_id             str
order_date              str
product_category        str
amount              float64
discount_pct        float64
is_returned           int64
dtype: object

Missing values:
order_id             0
customer_id          0
order_date           0
product_category     0
amount              10
discount_pct         8
is_returned          0
dtype: int64


In [12]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df)) * 100
})

missing_summary

,missing_count,missing_percentage
order_id,0,0.0
customer_id,0,0.0
order_date,0,0.0
product_category,0,0.0
amount,10,2.0
discount_pct,8,1.6
is_returned,0,0.0


## Part 1: Data Wrangling

In this step, I convert the order date column into a proper datetime format, create the `net_amount` column, and filter the dataset to include only orders from the last 6 months of the dataset.

I use the maximum order date in the dataset as the reference date instead of today's date because the dataset may not contain current real-time orders.

In [13]:
# Convert order_date to datetime format
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

# Check date conversion
print("Data type of order_date:", df["order_date"].dtype)
print("Invalid dates after conversion:", df["order_date"].isnull().sum())

print("Minimum order date:", df["order_date"].min())
print("Maximum order date:", df["order_date"].max())

Data type of order_date: datetime64[us]
Invalid dates after conversion: 0
Minimum order date: 2023-07-01 00:00:00
Maximum order date: 2024-06-30 00:00:00


In [14]:
# Create net_amount column
df["net_amount"] = df["amount"] * (1 - df["discount_pct"] / 100)

# Display sample rows to verify the calculation
df[["order_id", "amount", "discount_pct", "net_amount"]].head(10)

,order_id,amount,discount_pct,net_amount
0,ORD_00180,15.15,25.0,11.3625
1,ORD_00303,117.31,10.0,105.5790
2,ORD_00421,643.68,25.0,482.7600
3,ORD_00163,134.04,10.0,120.6360
4,ORD_00032,17.79,50.0,8.8950
5,ORD_00231,646.11,0.0,646.1100
6,ORD_00391,198.29,15.0,168.5465
7,ORD_00367,770.89,0.0,770.8900
8,ORD_00432,157.07,20.0,125.6560
9,ORD_00150,70.16,10.0,63.1440


In [15]:
# Find the latest order date in the dataset
max_order_date = df["order_date"].max()

# Calculate cutoff date: 6 months before the latest order date
cutoff_date = max_order_date - pd.DateOffset(months=6)

# Filter orders placed in the last 6 months of the dataset
df_last_6_months = df[df["order_date"] >= cutoff_date].copy()

print("Maximum order date in dataset:", max_order_date)
print("Cutoff date for last 6 months:", cutoff_date)
print("Shape before filtering:", df.shape)
print("Shape after filtering:", df_last_6_months.shape)

print("Filtered minimum date:", df_last_6_months["order_date"].min())
print("Filtered maximum date:", df_last_6_months["order_date"].max())

Maximum order date in dataset: 2024-06-30 00:00:00
Cutoff date for last 6 months: 2023-12-30 00:00:00
Shape before filtering: (500, 8)
Shape after filtering: (233, 8)
Filtered minimum date: 2023-12-31 00:00:00
Filtered maximum date: 2024-06-30 00:00:00


In [16]:
# Find the latest order date in the dataset
max_order_date = df["order_date"].max()

# Calculate cutoff date: 6 months before the latest order date
cutoff_date = max_order_date - pd.DateOffset(months=6)

# Filter orders placed in the last 6 months of the dataset
df_last_6_months = df[df["order_date"] >= cutoff_date].copy()

print("Maximum order date in dataset:", max_order_date)
print("Cutoff date for last 6 months:", cutoff_date)
print("Shape before filtering:", df.shape)
print("Shape after filtering:", df_last_6_months.shape)

print("Filtered minimum date:", df_last_6_months["order_date"].min())
print("Filtered maximum date:", df_last_6_months["order_date"].max())

Maximum order date in dataset: 2024-06-30 00:00:00
Cutoff date for last 6 months: 2023-12-30 00:00:00
Shape before filtering: (500, 8)
Shape after filtering: (233, 8)
Filtered minimum date: 2023-12-31 00:00:00
Filtered maximum date: 2024-06-30 00:00:00


In [17]:
# Save the filtered dataset for later analysis
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "orders_last_6_months.csv"

df_last_6_months.to_csv(PROCESSED_PATH, index=False)

print("Processed dataset saved to:", PROCESSED_PATH)
print("File exists:", PROCESSED_PATH.exists())

Processed dataset saved to: d:\11.Running Projects\interview-return-analysis\data\processed\orders_last_6_months.csv
File exists: True


In [18]:
df_last_6_months.head()

,order_id,customer_id,order_date,product_category,amount,discount_pct,is_returned,net_amount
267,ORD_00045,CUST_057,2023-12-31,Food,24.41,NaN,0,NaN
268,ORD_00449,CUST_068,2023-12-31,Books,27.57,25.0,0,20.6775
269,ORD_00186,CUST_032,2024-01-02,Food,54.50,10.0,0,49.0500
270,ORD_00128,CUST_061,2024-01-02,Clothing,174.93,15.0,0,148.6905
271,ORD_00059,CUST_074,2024-01-02,Electronics,83.92,20.0,0,67.1360


In [19]:
df_last_6_months.isnull().sum()

order_id            0
customer_id         0
order_date          0
product_category    0
amount              5
discount_pct        2
is_returned         0
net_amount          7
dtype: int64